In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings

from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
OUTPUT_DIR = Path("../reports")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("07 — LEAKAGE ANALİZİ: Address / IP / Port Feature Removal")
print("=" * 60)

In [ ]:
FEATURED_PATH  = "../data/csv/featured_dataset.csv"
FEATURES_TXT   = "../data/csv/selected_features.txt"
LABEL_MAP_PATH = "../data/csv/label_mapping.csv"

# Feature listesini oku (04_modeling ile aynı yöntem)
with open(FEATURES_TXT) as f:
    FEATURE_COLS = [line.strip() for line in f if line.strip()]

label_mapping = pd.read_csv(LABEL_MAP_PATH)
INT_TO_LABEL  = dict(zip(label_mapping["label_int"], label_mapping["label_string"]))

df = pd.read_csv(FEATURED_PATH, low_memory=False)

print(f"Dataset: {df.shape[0]:,} satır, {df.shape[1]} sütun")
print(f"Selected features: {len(FEATURE_COLS)} adet")
print(f"Sınıf sayısı: {df['label_multiclass'].nunique()}")

In [ ]:
# Leakage pattern'leri — CIC-IDS2017'ye özel
LEAKAGE_PATTERNS = [
    r"source.?address", r"destination.?address",
    r"src.?ip", r"dst.?ip",
    r"source.?ip", r"destination.?ip",
    r"flow.?id",
    r"timestamp",
]

# Destination Port — özel durum (model'de önemli ama leakage riski var)
PORT_PATTERNS = [
    r"^destination.?port$",
]

def find_matching_columns(columns, patterns):
    """Verilen pattern'lere uyan kolon isimlerini bulur."""
    found = []
    for col in columns:
        col_norm = col.lower().strip().replace(" ", "_")
        for pat in patterns:
            if re.search(pat, col_norm):
                found.append(col)
                break
    return sorted(set(found))

# Tespit
leakage_cols = find_matching_columns(FEATURE_COLS, LEAKAGE_PATTERNS)
port_cols    = find_matching_columns(FEATURE_COLS, PORT_PATTERNS)
all_remove   = sorted(set(leakage_cols + port_cols))

print("=" * 60)
print("LEAKAGE RİSKİ TESPİTİ")
print("=" * 60)
print(f"\nAddress/IP tabanlı leakage kolonları: {leakage_cols if leakage_cols else 'YOK'}")
print(f"Port tabanlı leakage kolonları:       {port_cols if port_cols else 'YOK'}")
print(f"\nToplam kaldırılacak: {len(all_remove)} feature")
for col in all_remove:
    print(f"  - {col}")

if not port_cols:
    print("\n[BİLGİ] 'Destination Port' selected_features.txt içinde")
    # Doğrudan kontrol
    dp_check = [c for c in FEATURE_COLS if "destination" in c.lower() and "port" in c.lower()]
    if dp_check:
        print(f"  Bulundu: {dp_check} — kaldırılacak listeye ekleniyor.")
        all_remove = sorted(set(all_remove + dp_check))
    else:
        print("  Destination Port feature listesinde bulunamadı.")
        print("  03_feature_engineering'de korelasyon/MI filtresi ile çıkarılmış olabilir.")

In [ ]:
def train_evaluate_rf(X, y, experiment_name, int_to_label=None):
    """
    RF eğit, değerlendir, metrikleri döndür.
    04_modeling ile aynı parametreler kullanılıyor.
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
    )

    model = RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=2  # 04_modeling ile tutarlı
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    metrics = {
        "experiment":   experiment_name,
        "n_features":   X.shape[1],
        "accuracy":     accuracy_score(y_test, y_pred),
        "precision_w":  precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "recall_w":     recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "f1_weighted":  f1_score(y_test, y_pred, average="weighted", zero_division=0),
        "f1_macro":     f1_score(y_test, y_pred, average="macro", zero_division=0),
    }
    return model, metrics, y_test, y_pred


def plot_confusion_matrix(y_true, y_pred, title, save_path, int_to_label=None):
    """Confusion matrix çiz ve kaydet."""
    labels = sorted(set(y_true) | set(y_pred))
    if int_to_label:
        display_labels = [int_to_label.get(l, str(l)) for l in labels]
    else:
        display_labels = [str(l) for l in labels]

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    fig.suptitle(title, fontsize=13, fontweight="bold")

    for ax, data, fmt, subtitle in [
        (axes[0], cm,      ",d",  "Ham Sayılar"),
        (axes[1], cm_norm, ".2f", "Normalize (Recall bazında)")
    ]:
        sns.heatmap(data, annot=True, fmt=fmt, cmap="Blues",
                    xticklabels=display_labels, yticklabels=display_labels,
                    ax=ax, annot_kws={"size": 7})
        ax.set_xlabel("Tahmin Edilen")
        ax.set_ylabel("Gerçek")
        ax.set_title(subtitle)
        ax.tick_params(axis='x', rotation=45, labelsize=7)
        ax.tick_params(axis='y', rotation=0,  labelsize=7)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✓ Kaydedildi: {save_path}")

In [ ]:
print("=" * 60)
print("DENEY 1: Tüm feature'lar dahil (BEFORE leakage removal)")
print("=" * 60)

X_before = df[FEATURE_COLS].values
y_mc     = df["label_multiclass"].values

_, metrics_before, yt_b, yp_b = train_evaluate_rf(
    X_before, y_mc, "BEFORE_leakage_removal", INT_TO_LABEL
)

print(f"\n  Feature sayısı : {metrics_before['n_features']}")
print(f"  Accuracy       : {metrics_before['accuracy']:.4f}")
print(f"  Precision (w)  : {metrics_before['precision_w']:.4f}")
print(f"  Recall (w)     : {metrics_before['recall_w']:.4f}")
print(f"  F1 Weighted    : {metrics_before['f1_weighted']:.4f}")
print(f"  F1 Macro       : {metrics_before['f1_macro']:.4f}")

plot_confusion_matrix(
    yt_b, yp_b,
    "Confusion Matrix — BEFORE Leakage Removal (Multiclass)",
    OUTPUT_DIR / "confusion_leakage_before.png",
    INT_TO_LABEL
)

In [ ]:
print("=" * 60)
print("DENEY 2: Address/IP/Port feature'lar çıkarıldı (AFTER)")
print("=" * 60)

# Sadece var olan kolonları çıkar
features_after = [f for f in FEATURE_COLS if f not in all_remove]
actually_removed = [f for f in all_remove if f in FEATURE_COLS]
not_found        = [f for f in all_remove if f not in FEATURE_COLS]

print(f"  Çıkarılan feature'lar ({len(actually_removed)}):")
for f in actually_removed:
    print(f"    - {f}")
if not_found:
    print(f"  Zaten listede olmayan ({len(not_found)}):")
    for f in not_found:
        print(f"    - {f}")

print(f"\n  Kalan feature sayısı: {len(features_after)}")

X_after = df[features_after].values

_, metrics_after, yt_a, yp_a = train_evaluate_rf(
    X_after, y_mc, "AFTER_leakage_removal", INT_TO_LABEL
)

print(f"\n  Accuracy       : {metrics_after['accuracy']:.4f}")
print(f"  Precision (w)  : {metrics_after['precision_w']:.4f}")
print(f"  Recall (w)     : {metrics_after['recall_w']:.4f}")
print(f"  F1 Weighted    : {metrics_after['f1_weighted']:.4f}")
print(f"  F1 Macro       : {metrics_after['f1_macro']:.4f}")

plot_confusion_matrix(
    yt_a, yp_a,
    "Confusion Matrix — AFTER Leakage Removal (Multiclass)",
    OUTPUT_DIR / "confusion_leakage_after.png",
    INT_TO_LABEL
)

In [ ]:
print("=" * 60)
print("LEAKAGE KARŞILAŞTIRMA TABLOSU")
print("=" * 60)

comp_leakage = pd.DataFrame([metrics_before, metrics_after])
comp_leakage["delta_accuracy"]    = comp_leakage["accuracy"]    - metrics_before["accuracy"]
comp_leakage["delta_f1_weighted"] = comp_leakage["f1_weighted"] - metrics_before["f1_weighted"]
comp_leakage["delta_f1_macro"]    = comp_leakage["f1_macro"]    - metrics_before["f1_macro"]

display_cols = ["experiment", "n_features", "accuracy", "precision_w", "recall_w",
                "f1_weighted", "f1_macro", "delta_accuracy", "delta_f1_weighted", "delta_f1_macro"]

print(comp_leakage[display_cols].round(4).to_string(index=False))

# CSV olarak kaydet
comp_leakage.to_csv(OUTPUT_DIR / "leakage_comparison.csv", index=False)
print(f"\n✓ Kaydedildi: {OUTPUT_DIR / 'leakage_comparison.csv'}")